# 🎵 Waveform + Mel Spectrogram 앙상블 모델 (3가지 상태 분류)

이 노트북에서는 Waveform과 Mel Spectrogram을 모두 사용하여 앙상블 모델을 구축하고, Vote 형식으로 예측합니다.
**멀티헤드 모델의 첫 번째 헤드로 사용할 수 있도록 구성**되었습니다.

## 📋 목차
1. **데이터 로드 및 Combined 데이터 제거**
2. **멜스펙트로그램 특징 추출 및 마스크 강조**
3. **Waveform 기반 모델**: 1D CNN 모델
4. **Mel Spectrogram 기반 모델**: 마스킹 CNN 모델
5. **앙상블 모델**: 두 모델의 예측을 Vote로 결합
6. **3가지 상태 분류**: Braking, Idle, Startup
7. **테스트 데이터 증강**: 3개 상태 컬럼 개수 맞추기

In [ ]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================

import os
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict
import librosa
import librosa.display

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# 머신러닝
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 공통 유틸리티
from utils import setup_plotting, get_data_dir, get_state_mapping, get_state_names

# 프로젝트 모듈
from app.ml.features.extractor import AudioFeatureExtractor, AudioConfig
from app.ml.features.augmentation import AudioAugmentor, AugmentationConfig
from app.ml.training.trainer import Trainer, create_optimizer, create_scheduler

# 시각화 설정
setup_plotting()

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 라이브러리 로드 완료!")
print(f"🖥️ Device: {device}")

---
## 1. 데이터 로드 및 Combined 데이터 제거

In [ ]:
# ============================================================
# 데이터 로드 및 Combined 데이터 제거
# ============================================================

data_dir = get_data_dir()
state_names = get_state_names()

# 오디오 파일 수집 (combined 제외)
all_files = []
all_labels = []

for state_dir in sorted(data_dir.iterdir()):
    if not state_dir.is_dir() or state_dir.name == 'augmented':
        continue
    
    state_name = state_dir.name
    
    for problem_dir in sorted(state_dir.iterdir()):
        if not problem_dir.is_dir():
            continue
        
        problem_name = problem_dir.name
        
        # Combined 클래스 제외 (하위 폴더에 combined가 포함된 경우 제외)
        if 'combined' in problem_name.lower():
            continue
        
                # 하위 폴더 확인 (combined 제외)
        has_combined = False
        try:
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and 'combined' in sub_dir.name.lower():
                    has_combined = True
                    break
        except:
            pass
        
        if has_combined:
            continue
        
        # WAV 파일 수집
        wav_files = list(problem_dir.glob('*.wav'))
        
        for wav_file in wav_files:
            all_files.append(wav_file)
            all_labels.append(f"{state_name}_{problem_name}")
        
        # 하위 폴더의 WAV 파일 수집 (combined 제외)
        try:
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and 'combined' not in sub_dir.name.lower():
                    sub_wav_files = list(sub_dir.glob('*.wav'))
                    for wav_file in sub_wav_files:
                        all_files.append(wav_file)
                        all_labels.append(f"{state_name}_{problem_name}/{sub_dir.name}")
        except:
            pass

print(f"📊 총 데이터: {len(all_files)}개")
print(f"📊 상태별 분포:")

# 상태별 분포 확인
state_counts = Counter([label.split('_')[0] for label in all_labels])
for state, count in state_counts.items():
    print(f"  {state}: {count}개")

---
## 2. 멜스펙트로그램 특징 추출 및 마스크 강조

In [ ]:
# ============================================================
# 오디오 특징 추출 설정
# ============================================================

audio_config = AudioConfig(
    sample_rate=22050,
    duration=5.0,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)

feature_extractor = AudioFeatureExtractor(audio_config)

print("🔄 멜스펙트로그램 및 Waveform 추출 중...")

# 멜스펙트로그램 및 Waveform 추출
X_mel = []
X_waveform = []
y_states = []

for file_path, label in tqdm(zip(all_files, all_labels), total=len(all_files)):
    try:
        # 오디오 로드
        y, sr = librosa.load(str(file_path), sr=audio_config.sample_rate)
        
        # Waveform 추출 (패딩 포함)
        target_length = int(audio_config.sample_rate * audio_config.duration)
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        elif len(y) > target_length:
            y = y[:target_length]
        
        X_waveform.append(y)
        
        # Mel Spectrogram 추출
        mel_spec = feature_extractor.extract_mel_spectrogram(y, sr)
        X_mel.append(mel_spec)
        
        # 상태 레이블 추출
        state = label.split('_')[0]
        y_states.append(state)
        
    except Exception as e:
        print(f"⚠️ 오류 발생: {file_path} - {e}")
        continue

X_mel = np.array(X_mel)
X_waveform = np.array(X_waveform)
y_states = np.array(y_states)

print(f"✅ 추출 완료!")
print(f"  Mel Spectrogram shape: {X_mel.shape}")
print(f"  Waveform shape: {X_waveform.shape}")
print(f"  상태 레이블 shape: {y_states.shape}")

In [ ]:
# ============================================================
# 마스크 강조 (중요 영역 강조)
# ============================================================

# 상태별 평균 스펙트로그램 계산
state_means = {}
for state in np.unique(y_states):
    state_indices = np.where(y_states == state)[0]
    state_means[state] = np.mean(X_mel[state_indices], axis=0)

# 중요 영역 마스크 생성 (각 상태별로 차이가 큰 영역)
importance_mask = np.zeros_like(X_mel[0])

states = list(state_means.keys())
for i in range(len(states)):
    for j in range(i + 1, len(states)):
        diff = np.abs(state_means[states[i]] - state_means[states[j]])
        importance_mask = np.maximum(importance_mask, diff)

# 정규화
importance_mask = (importance_mask - importance_mask.min()) / (importance_mask.max() - importance_mask.min() + 1e-8)

# 마스크를 텐서로 변환
importance_mask_tensor = torch.FloatTensor(importance_mask).unsqueeze(0).to(device)

print(f"✅ 중요 영역 마스크 생성 완료!")
print(f"  마스크 shape: {importance_mask.shape}")
print(f"  마스크 범위: [{importance_mask.min():.3f}, {importance_mask.max():.3f}]")

---
## 3. 모델 정의

In [ ]:
# ============================================================
# Waveform 1D CNN 모델
# ============================================================

class WaveformCNN1D(nn.Module):
    """Waveform을 입력으로 받는 1D CNN 모델"""
    
    def __init__(
        self,
        num_classes: int,
        input_length: int = 110250,  # 5초 @ 22050 Hz
        base_channels: int = 64,
        dropout: float = 0.3
    ):
        super().__init__()
        
        self.num_classes = num_classes
        
        # 1D Convolutional layers
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(base_channels * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc_out = nn.Linear(128, num_classes)
        
    def forward(self, x):
        # x: (batch, 1, length)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

print("✅ WaveformCNN1D 모델 정의 완료!")

In [ ]:
# ============================================================
# 마스킹 기반 Mel Spectrogram CNN 모델
# ============================================================

class MaskedSpatialAttention(nn.Module):
    """마스킹 기반 Spatial Attention"""
    
    def __init__(self, importance_mask: torch.Tensor, learnable: bool = True):
        super().__init__()
        self.importance_mask = nn.Parameter(importance_mask, requires_grad=learnable)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, channels, freq, time)
        # 마스크를 입력 shape에 맞게 조정
        mask = self.importance_mask.expand(x.size(0), -1, -1)
        mask = mask.unsqueeze(1)  # (batch, 1, freq, time)
        
        # 마스크 적용 (중요 영역 강조)
        x = x * (1 + mask)
        return x


class MaskedCNN(nn.Module):
    """마스킹 기반 Mel Spectrogram CNN 모델"""
    
    def __init__(self, num_classes: int, importance_mask: torch.Tensor, 
                 in_channels: int = 1, base_channels: int = 32, dropout: float = 0.3):
        super().__init__()
        
        self.num_classes = num_classes
        
        # 마스킹 기반 Spatial Attention
        self.masked_attention = MaskedSpatialAttention(importance_mask, learnable=True)
        
        # Convolutional layers
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, padding=1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(base_channels * 8, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc_out = nn.Linear(128, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 마스킹 기반 Attention 적용
        x = self.masked_attention(x)
        
        # Convolutional blocks
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        
        # Global Average Pooling
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        
        return x

print("✅ MaskedCNN 모델 정의 완료!")

---
## 4. 앙상블 모델 (Vote 방식)

In [ ]:
# ============================================================
# 앙상블 모델 클래스 (Vote 방식)
# ============================================================

class EnsembleVoteModel(nn.Module):
    """
    Waveform과 Mel Spectrogram 모델을 Vote 방식으로 결합
    멀티헤드 모델의 첫 번째 헤드로 사용 가능
    """
    
    def __init__(
        self,
        waveform_model: nn.Module,
        spectrogram_model: nn.Module,
        vote_method: str = 'soft'  # 'hard' or 'soft'
    ):
        super().__init__()
        
        self.waveform_model = waveform_model
        self.spectrogram_model = spectrogram_model
        self.vote_method = vote_method
    
    def forward(self, waveform_input, spectrogram_input):
        """
        두 모델의 예측을 결합
        
        Args:
            waveform_input: Waveform 텐서 (batch, 1, length)
            spectrogram_input: Mel Spectrogram 텐서 (batch, 1, freq, time)
        
                Returns:
            ensemble_output: 앙상블 예측 결과 (batch, num_classes) - logits
        """
        # Waveform 모델 예측
        waveform_output = self.waveform_model(waveform_input)
        waveform_probs = F.softmax(waveform_output, dim=1)
        
        # Mel Spectrogram 모델 예측
        spec_output = self.spectrogram_model(spectrogram_input)
        spec_probs = F.softmax(spec_output, dim=1)
        
                # Vote 방식에 따라 결합
        if self.vote_method == 'hard':
            # Hard Voting: 다수결
            waveform_pred = waveform_output.argmax(dim=1)
            spec_pred = spec_output.argmax(dim=1)
            
            ensemble_pred = torch.stack([waveform_pred, spec_pred], dim=1)
            ensemble_pred = torch.mode(ensemble_pred, dim=1)[0]
            
            # One-hot으로 변환 후 logits로 변환
            ensemble_output = F.one_hot(ensemble_pred, num_classes=waveform_output.size(1)).float()
            # logits로 변환 (큰 값 사용)
            ensemble_output = ensemble_output * 10.0 - 5.0
        
        else:  # soft voting
            # Soft Voting: logits 평균 (학습 시 사용)
            ensemble_output = (waveform_output + spec_output) / 2
        
        return ensemble_output

print("✅ EnsembleVoteModel 클래스 정의 완료!")

---
## 5. 데이터셋 클래스

In [ ]:
# ============================================================
# 데이터셋 클래스
# ============================================================

class StateClassificationDataset(Dataset):
    """3가지 상태 분류용 데이터셋"""
    
    def __init__(self, X_waveform, X_mel, y_states, state_to_idx):
        self.X_waveform = X_waveform
        self.X_mel = X_mel
        self.y_states = y_states
        self.state_to_idx = state_to_idx
        
        # 레이블을 인덱스로 변환
        self.y_indices = np.array([state_to_idx[state] for state in y_states])
    
    def __len__(self):
        return len(self.X_waveform)
    
    def __getitem__(self, idx):
        waveform = torch.FloatTensor(self.X_waveform[idx]).unsqueeze(0)  # (1, length)
        mel_spec = torch.FloatTensor(self.X_mel[idx]).unsqueeze(0)  # (1, freq, time)
        label = torch.LongTensor([self.y_indices[idx]])[0]
        
        return waveform, mel_spec, label

print("✅ StateClassificationDataset 클래스 정의 완료!")

---
## 6. 데이터 분할 및 준비

In [ ]:
# ============================================================
# 상태 레이블 매핑
# ============================================================

unique_states = sorted(np.unique(y_states))
state_to_idx = {state: idx for idx, state in enumerate(unique_states)}
idx_to_state = {idx: state for state, idx in state_to_idx.items()}
num_classes = len(unique_states)

print(f"📊 상태 클래스:")
for state, idx in state_to_idx.items():
    count = np.sum(y_states == state)
    print(f"  {state}: {count}개 (인덱스: {idx})")

print(f"\n총 클래스 수: {num_classes}개")

In [ ]:
# ============================================================
# Train/Val/Test 분할
# ============================================================

# 먼저 Train/Test 분할 (80:20)
X_waveform_train, X_waveform_test, X_mel_train, X_mel_test, y_train, y_test = train_test_split(
    X_waveform, X_mel, y_states, test_size=0.2, random_state=42, stratify=y_states
)

# Train을 다시 Train/Val 분할 (80:20)
X_waveform_train, X_waveform_val, X_mel_train, X_mel_val, y_train, y_val = train_test_split(
    X_waveform_train, X_mel_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"📊 데이터 분할:")
print(f"  Train: {len(X_waveform_train)}개")
print(f"  Val: {len(X_waveform_val)}개")
print(f"  Test: {len(X_waveform_test)}개")

In [ ]:
# ============================================================
# 데이터셋 및 DataLoader 생성
# ============================================================

train_dataset = StateClassificationDataset(X_waveform_train, X_mel_train, y_train, state_to_idx)
val_dataset = StateClassificationDataset(X_waveform_val, X_mel_val, y_val, state_to_idx)
test_dataset = StateClassificationDataset(X_waveform_test, X_mel_test, y_test, state_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("✅ 데이터셋 및 DataLoader 생성 완료!")

---
## 7. 모델 생성 및 학습

In [ ]:
# ============================================================
# 개별 모델 생성
# ============================================================

# Waveform 모델
waveform_model = WaveformCNN1D(
    num_classes=num_classes,
    input_length=X_waveform.shape[1],
    base_channels=64,
    dropout=0.3
).to(device)

# Mel Spectrogram 모델
mel_model = MaskedCNN(
    num_classes=num_classes,
    importance_mask=importance_mask_tensor,
    in_channels=1,
    base_channels=32,
    dropout=0.3
).to(device)

# 앙상블 모델
ensemble_model = EnsembleVoteModel(
    waveform_model=waveform_model,
    spectrogram_model=mel_model,
    vote_method='soft'
).to(device)

print("✅ 모델 생성 완료!")
print(f"  Waveform 모델 파라미터: {sum(p.numel() for p in waveform_model.parameters()):,}개")
print(f"  Mel Spectrogram 모델 파라미터: {sum(p.numel() for p in mel_model.parameters()):,}개")
print(f"  앙상블 모델 파라미터: {sum(p.numel() for p in ensemble_model.parameters()):,}개")

In [ ]:
# ============================================================
# 학습 함수
# ============================================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for waveform, mel_spec, labels in train_loader:
        waveform = waveform.to(device)
        mel_spec = mel_spec.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
                # 앙상블 모델 예측 (logits 반환)
        outputs = model(waveform, mel_spec)
        
        # CrossEntropyLoss는 logits를 기대함
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    
    return total_loss / len(train_loader), correct / total


def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for waveform, mel_spec, labels in val_loader:
            waveform = waveform.to(device)
            mel_spec = mel_spec.to(device)
            labels = labels.to(device)
            
                        outputs = model(waveform, mel_spec)
            
            # CrossEntropyLoss는 logits를 기대함
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            
            total_loss += loss.item()
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(val_loader), correct / total, all_preds, all_labels

print("✅ 학습/검증 함수 정의 완료!")

In [ ]:
# ============================================================
# 모델 학습
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(ensemble_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

num_epochs = 50
best_val_acc = 0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("🚀 앙상블 모델 학습 시작\n")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(ensemble_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = validate(ensemble_model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
        if val_acc > best_val_acc:
        best_val_acc = val_acc
        model_path = Path('checkpoints')
        model_path.mkdir(exist_ok=True)
        torch.save(ensemble_model.state_dict(), model_path / 'best_ensemble_state_model.pth')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print()

print("✅ 학습 완료!")

---
## 8. 테스트 데이터 증강 (3개 상태 컬럼 개수 맞추기)

In [ ]:
# ============================================================
# 테스트 데이터 상태별 개수 확인
# ============================================================

test_state_counts = Counter(y_test)
print("📊 테스트 데이터 상태별 개수:")
for state, count in sorted(test_state_counts.items()):
    print(f"  {state}: {count}개")

# 가장 많은 개수 찾기
max_count = max(test_state_counts.values())
print(f"\n최대 개수: {max_count}개")

In [ ]:
# ============================================================
# 데이터 증강 설정
# ============================================================

augmentation_config = AugmentationConfig(
    time_shift=True,
    pitch_shift=True,
    time_stretch=True,
    add_noise=True
)

augmentor = AudioAugmentor(augmentation_config)

print("✅ 데이터 증강 설정 완료!")

In [ ]:
# ============================================================
# 테스트 데이터 증강 (3개 상태 컬럼 개수 맞추기)
# ============================================================

# 각 상태별로 필요한 증강 개수 계산
augmented_waveforms = []
augmented_mels = []
augmented_labels = []

# 테스트 데이터를 상태별로 분리
test_data_by_state = {state: [] for state in unique_states}
for i, state in enumerate(y_test):
    test_data_by_state[state].append(i)

# 각 상태별로 증강 수행
for state in unique_states:
    indices = test_data_by_state[state]
    current_count = len(indices)
    
    # 목표 개수: 가장 많은 상태의 개수 (하지만 그 상태도 조금 증강)
    if current_count == max_count:
        # 가장 많은 상태도 조금 증강 (10% 정도)
        target_count = int(max_count * 1.1)
    else:
        # 다른 상태는 max_count에 맞춤
        target_count = max_count
    
    needed = target_count - current_count
    
    if needed > 0:
        print(f"\n🔄 {state} 상태 증강 중... (현재: {current_count}개, 목표: {target_count}개, 필요: {needed}개)")
        
        # 증강할 샘플 선택 (반복 사용)
        aug_count = 0
        while aug_count < needed:
            for idx in indices:
                if aug_count >= needed:
                    break
                
                # 원본 데이터 가져오기
                waveform = X_waveform_test[idx].copy()
                mel_spec = X_mel_test[idx].copy()
                
                # 증강 적용
                augmented_waveform = augmentor.augment(waveform, audio_config.sample_rate)
                
                # Mel Spectrogram 재계산
                augmented_mel = feature_extractor.extract_mel_spectrogram(
                    augmented_waveform, audio_config.sample_rate
                )
                
                augmented_waveforms.append(augmented_waveform)
                augmented_mels.append(augmented_mel)
                augmented_labels.append(state)
                
                aug_count += 1

# 증강된 데이터를 원본 테스트 데이터에 추가
if len(augmented_waveforms) > 0:
    X_waveform_test_aug = np.concatenate([X_waveform_test, np.array(augmented_waveforms)], axis=0)
    X_mel_test_aug = np.concatenate([X_mel_test, np.array(augmented_mels)], axis=0)
    y_test_aug = np.concatenate([y_test, np.array(augmented_labels)], axis=0)
else:
    # 증강이 필요 없는 경우 원본 사용
    X_waveform_test_aug = X_waveform_test
    X_mel_test_aug = X_mel_test
    y_test_aug = y_test

print(f"\n✅ 증강 완료!")
print(f"  원본 테스트 데이터: {len(X_waveform_test)}개")
print(f"  증강된 데이터: {len(augmented_waveforms)}개")
print(f"  총 테스트 데이터: {len(X_waveform_test_aug)}개")

# 증강 후 상태별 개수 확인
aug_state_counts = Counter(y_test_aug)
print(f"\n📊 증강 후 테스트 데이터 상태별 개수:")
for state, count in sorted(aug_state_counts.items()):
    print(f"  {state}: {count}개")

In [ ]:
# ============================================================
# 증강된 테스트 데이터셋 생성
# ============================================================

test_dataset_aug = StateClassificationDataset(
    X_waveform_test_aug, X_mel_test_aug, y_test_aug, state_to_idx
)
test_loader_aug = DataLoader(test_dataset_aug, batch_size=32, shuffle=False)

print("✅ 증강된 테스트 데이터셋 생성 완료!")

---
## 9. 최종 평가

In [ ]:
# ============================================================
# 최종 테스트 평가
# ============================================================

# 최고 모델 로드
model_path = Path('checkpoints') / 'best_ensemble_state_model.pth'
if model_path.exists():
    ensemble_model.load_state_dict(torch.load(model_path))
    print(f"✅ 모델 로드 완료: {model_path}")
else:
    print(f"⚠️ 모델 파일을 찾을 수 없습니다: {model_path}")

# 테스트 평가
test_loss, test_acc, test_preds, test_labels = validate(
    ensemble_model, test_loader_aug, criterion, device
)

print(f"\n📊 최종 테스트 결과:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")

# 분류 리포트
print(f"\n📋 분류 리포트:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=[idx_to_state[i] for i in range(num_classes)]
))

# 혼동 행렬
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[idx_to_state[i] for i in range(num_classes)],
            yticklabels=[idx_to_state[i] for i in range(num_classes)])
plt.title('Confusion Matrix - Ensemble State Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
## 10. 멀티헤드 모델용 출력 형식

이 모델은 멀티헤드 모델의 첫 번째 헤드로 사용할 수 있도록 설계되었습니다.
다음과 같이 사용할 수 있습니다:

```python
# 멀티헤드 모델에서 사용 예시
class MultiHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 첫 번째 헤드: 상태 분류 (이 앙상블 모델)
        self.state_classifier = ensemble_model
        # 두 번째 헤드: 문제 분류 (다른 모델)
        # ...
    
    def forward(self, waveform, mel_spec):
        # 상태 분류
        state_output = self.state_classifier(waveform, mel_spec)
        # ...
        return state_output, ...
```